In [24]:
import re
import pandas as pd

In [25]:
def clean(text):
    # "hỗ trợ chụp 24MP hoặc 48MP"
    text = re.sub(r'hỗ trợ chụp.*?(?=\D{3}|$)', '', text, flags=re.IGNORECASE)
    # "(24MP và 48MP)", "(24MP hoặc 48MP)"
    text = re.sub(r'\([\d.]+\s*MP\s*(?:và|hoặc|or)\s*[\d.]+\s*MP\)', '', text, flags=re.IGNORECASE)
    # "hoặc 48MP" còn sót
    text = re.sub(r'(?:hoặc|hoac|hay|or)\s+[\d.]+\s*MP', '', text, flags=re.IGNORECASE)
    return text

In [26]:
def extract_mp_values(text):
    text = clean(text)
    vals  = re.findall(r'([\d.]+)\s*(?:MP|megapixel)', text, re.IGNORECASE)
    vals += re.findall(r'([\d.]+)M(?=[^a-zA-Z]|$)', text)
    return [float(v) for v in vals if v.count('.') <= 1 and float(v) >= 0.3]

In [27]:
def extract_aperture(text):
    vals   = re.findall(r'[fƒ]\s*/?\s*([\d.]+)', text, re.IGNORECASE)
    floats = [float(v) for v in vals if v.count('.') <= 1 and 0.5 <= float(v) <= 6.0]
    return min(floats) if floats else 0

In [28]:
def count_cameras(text: str, mps: list):
    if len(mps) >= 2:
        return len(mps)
    m = re.search(r'(\d)\s*camera', text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return 1 if mps else 0

In [29]:
def parse_rear(text):
    if not isinstance(text, str) or not text.strip():
        return {"rear_count": 0, "rear_mp_max": 0, "rear_f/": 0, "rear_ois": 0, "rear_telephoto": 0, "rear_wide": 0}
    mps      = extract_mp_values(text)
    aperture = extract_aperture(text)
    return {
        "rear_count": count_cameras(text, mps),
        "rear_mp_max": max(mps) if mps else 0,
        "rear_f/": aperture if aperture else 0,
        "rear_ois": int(bool(re.search(r'\bOIS\b', text, re.IGNORECASE))),
        "rear_telephoto": int(bool(re.search(r'tele(?:photo)?|zoom quang|kính tiềm vọng|periscope', text, re.IGNORECASE))),
        "rear_wide": int(bool(re.search(r'siêu rộng|ultra.?wide|góc rộng|wide|superwide', text, re.IGNORECASE))),
    }

In [30]:
def parse_front(text):
    if not isinstance(text, str) or not text.strip():
        return {"front_mp": 0, "front_f/": 0}
    mps = extract_mp_values(text)
    aperture = extract_aperture(text)
    return {
        "front_mp": max(mps) if mps else 0,
        "front_f/": aperture if aperture else 0,
    }

In [31]:
def parse_camera(df):
    rear  = df["Camera sau"].apply(parse_rear).apply(pd.Series)
    front = df["Camera trước"].apply(parse_front).apply(pd.Series)
    df_out = pd.concat([df, rear, front], axis=1)

    return df_out

In [32]:
df = pd.read_csv(r'cellphones_full.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

In [35]:
df = parse_camera(df)

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tên                    966 non-null    str    
 1   Giá                    966 non-null    str    
 2   Link                   966 non-null    str    
 3   Kích thước màn hình    865 non-null    str    
 4   Công nghệ màn hình     806 non-null    str    
 5   Camera sau             847 non-null    str    
 6   Camera trước           817 non-null    str    
 7   Chipset                850 non-null    str    
 8   Công nghệ NFC          763 non-null    str    
 9   Bộ nhớ trong           913 non-null    str    
 10  Thẻ SIM                695 non-null    str    
 11  Hệ điều hành           756 non-null    str    
 12  Độ phân giải màn hình  657 non-null    str    
 13  Tính năng màn hình     725 non-null    str    
 14  Loại CPU               586 non-null    str    
 15  Dung lượng RAM   

In [37]:
df[["Tên", "rear_count", "rear_mp_max", "rear_f/", "rear_ois", "rear_telephoto", "rear_wide", "front_mp", "front_f/"]].head(10)

,Tên,rear_count,rear_mp_max,rear_f/,rear_ois,rear_telephoto,rear_wide,front_mp,front_f/
0,iphone 17,3.0,48.0,1.6,1.0,1.0,1.0,18.0,1.9
1,oppo find x9s,3.0,50.0,0.0,0.0,1.0,1.0,32.0,0.0
2,iphone 17 promax,3.0,48.0,1.6,1.0,1.0,1.0,18.0,1.9
3,samsung galaxy s26,4.0,200.0,0.0,0.0,1.0,1.0,12.0,0.0
4,samsung galaxy s26,3.0,50.0,0.0,0.0,1.0,1.0,12.0,0.0
5,samsung galaxy s25,4.0,200.0,0.0,0.0,1.0,1.0,12.0,0.0
6,iphone 17,3.0,48.0,1.6,0.0,1.0,1.0,18.0,1.9
7,itel p55,1.0,50.0,0.0,0.0,0.0,1.0,8.0,0.0
8,oppo reno15 f,3.0,50.0,1.8,1.0,0.0,1.0,50.0,2.0
9,iphone 15,2.0,48.0,0.0,0.0,0.0,0.0,12.0,1.9
